# Ex 2 · Age-group sensitivity analysis

**Member C**

## Summary

In the 1,314-record sample, the unadjusted survival gap was **+7.54 percentage
points** (smokers minus non-smokers). Using the same age distribution
for both groups, the gap was **−4.18** with the baseline age bands, **+3.55**
with the coarse 50+ band, and **−2.17** with the finer older-age bands.
The adjusted gap was smaller in absolute size across all three schemes,
and its sign depended on the age boundaries.

## Context and methods

The baseline uses ages **18–29, 30–49, 50–64, and 65+**. Compare it with
the original **50+** grouping and a finer split at **50, 60, and 70**.
All three schemes use the same records and include their lower boundaries.

Survival rate is the share with `alive = 1` within each age band and
smoking status. Differences are in percentage points, smokers minus
non-smokers. For a single comparison per scheme, weight each age band's
difference by its share of the pooled sample. These estimates describe
the observed association.

## Data

Run from the repository root in the Pixi `smoke` kernel. Use
[the supplied CSV](data/smoking_data.csv) and
[D's preparation rules](smoking_is_bad.ipynb). Retain rows with missing
salary. Exclude rows missing age, smoking status, or outcome and report
the number excluded.

In [ ]:
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

raw = pd.read_csv(Path("data") / "smoking_data.csv")
prepared = raw.drop(columns="Unnamed: 0", errors="ignore").copy()
prepared["smokes"] = prepared["smoker"].map({"No": 0, "Yes": 1})
prepared["alive"] = prepared["outcome"].map({"Dead": 0, "Alive": 1})

for original, encoded in [("smoker", "smokes"), ("outcome", "alive")]:
    invalid = prepared[original].notna() & prepared[encoded].isna()
    if invalid.any():
        raise ValueError(f"Unexpected categories in {original}")

analysis = prepared.dropna(subset=["age", "smokes", "alive"]).copy()
print(f"Raw rows: {len(raw):,}")
print(f"Excluded for missing age, smoking status or outcome: {len(raw) - len(analysis):,}")
print(f"Analysis rows: {len(analysis):,}")
print(f"Age range: {analysis['age'].min()}–{analysis['age'].max()} years")

## Results

### Survival rates and sample sizes by age band

Each rate is reported with its numerator and denominator. Mean age is
included to check how similar the two groups are inside each band.

In [ ]:
open_end = float("inf")
schemes = {
    "Baseline": ([18, 30, 50, 65, open_end], ["18–29", "30–49", "50–64", "65+"]),
    "Coarse": ([18, 30, 50, open_end], ["18–29", "30–49", "50+"]),
    "Finer": (
        [18, 30, 50, 60, 70, open_end],
        ["18–29", "30–49", "50–59", "60–69", "70+"],
    ),
}


def summarize_by_age(data, edges, labels):
    age_band = pd.cut(data["age"], bins=edges, labels=labels, right=False)
    if age_band.isna().any():
        raise ValueError("The chosen age bands must cover every included age.")

    grouped = (
        data.assign(age_band=age_band)
        .groupby(["age_band", "smokes"], observed=False)
        .agg(n=("alive", "size"), alive=("alive", "sum"), mean_age=("age", "mean"))
        .unstack("smokes")
    )
    if not grouped["n"].gt(0).all().all():
        raise ValueError("Each age band needs observations in both smoking groups.")

    rates = grouped["alive"] / grouped["n"]
    return pd.DataFrame({
        "Non-smokers alive": grouped["alive"][0],
        "Non-smokers n": grouped["n"][0],
        "Non-smokers rate": rates[0],
        "Smokers alive": grouped["alive"][1],
        "Smokers n": grouped["n"][1],
        "Smokers rate": rates[1],
        "Mean age gap (years)": grouped["mean_age"][1] - grouped["mean_age"][0],
        "Survival gap (pp)": 100 * (rates[1] - rates[0]),
    })


tables = {
    name: summarize_by_age(analysis, edges, labels)
    for name, (edges, labels) in schemes.items()
}
all_bands = pd.concat(tables, names=["Scheme", "Age band"])
display(all_bands.style.format({
    "Non-smokers rate": "{:.1%}",
    "Smokers rate": "{:.1%}",
    "Mean age gap (years)": "{:+.2f}",
    "Survival gap (pp)": "{:+.2f}",
}))

### Overall differences under a common age distribution

The unadjusted estimate is the difference between the two overall rates.
Each age-standardized estimate uses the same pooled age weights for
both smoking groups within that scheme.

In [ ]:
overall_rates = analysis.groupby("smokes")["alive"].mean()
unadjusted_gap = 100 * (overall_rates.loc[1] - overall_rates.loc[0])
gaps = {"Unadjusted": unadjusted_gap}

for name, table in tables.items():
    pooled_counts = table["Non-smokers n"] + table["Smokers n"]
    assert pooled_counts.sum() == len(analysis)
    weights = pooled_counts / len(analysis)
    gaps[name] = (weights * table["Survival gap (pp)"]).sum()

gap_summary = pd.Series(gaps, name="Survival gap (pp)").to_frame()
display(gap_summary.style.format("{:+.2f}"))

In [ ]:
older_bands = {"Coarse": ["50+"], "Baseline": ["50–64", "65+"],
               "Finer": ["50–59", "60–69", "70+"]}
older_gaps = {
    f"{name} · {band}": tables[name].loc[band, "Survival gap (pp)"]
    for name, bands in older_bands.items()
    for band in bands
}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, layout="constrained")
fig.suptitle("Smoking and recorded survival in the Ex 2 sample")
for ax, title, values in [
    (axes[0], "A. Overall gap by scheme", gaps),
    (axes[1], "B. Within-band gaps at ages 50+", older_gaps),
]:
    y = range(len(values))
    ax.scatter(list(values.values()), y, color="#245f88", s=36, zorder=3)
    for position, gap in enumerate(values.values()):
        side = 1 if gap >= 0 else -1
        ax.annotate(
            f"{gap:+.2f}", (gap, position), xytext=(6 * side, 0),
            textcoords="offset points", ha="left" if side > 0 else "right",
            va="center", fontsize=9,
        )
    ax.axvline(0, color="#555b62", linewidth=1)
    ax.set_yticks(list(y), labels=list(values))
    ax.invert_yaxis()
    ax.set_title(title, loc="left")
    ax.set_xlabel("Smoker minus non-smoker (percentage points)")
    ax.grid(axis="x", color="#e3e7eb")
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)

axes[0].set_xlim(-12, 16)
with BytesIO() as image:
    fig.savefig(image, format="png", bbox_inches="tight")
    display(Image(
        data=image.getvalue(),
        alt="Two dot plots compare survival-rate gaps by age scheme and within bands at ages 50 and older. The overall age-standardized gap changes sign across schemes.",
    ))
plt.close(fig)

## Takeaways

- **The 50+ band combines people of quite different ages.** In that band,
  smokers were 5.4 years younger on average and had a survival gap of
  +11.63 percentage points. Splitting it at 65 reduced the within-band
  mean age differences to under half a year. The gaps were −8.12 points
  at ages 50–64 and −2.51 points at ages 65+.
- **Within-band results also depend on the cutoffs.** With the finer
  grouping, the gap at ages 60–69 was +2.16 points. The 70+ group had
  only 34 smokers, of whom 3 were alive, so its near-zero gap is imprecise.

These are descriptive comparisons of recorded survival in this sample.
The standardization accounts for age bands and leaves other potential
confounders unadjusted.